In [1]:
import pandas as pd
import numpy as np

base = "/scicore/home/schwede/barta0000/project/data/raw/"
meta_cols = ["sample", "source", "type", "KD[M]", "binder_type", "dataset"]

# train
eval_df = pd.read_csv(f"{base}/current/eval.csv")   # trees scale/direction invariant
y_train = eval_df["binder"].astype(int)
X_train = eval_df.drop(columns = meta_cols + ["binder"], errors="ignore")
feature_cols = X_train.columns.tolist()
print(f"train: {len(X_train)} rows, {X_train.shape[1]} features, classes={y_train.value_counts().to_dict()}")
X_train.head()

train: 71 rows, 49 features, classes={1: 44, 0: 27}


,af3_pae,af3_plddt,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq2,af3_lis,af3_iptm,af3_ptm,cf_af3_rmsd,...,boltz_template_ptm,boltz_template_iptm,boltz_template_plddt,boltz_template_iplddt,boltz_template_pde,boltz_template_ipde,boltz_template_per_chain_pair_iptm,boltz_template_ipsae,boltz_template_pdockq2,boltz_template_lis
0,2.972237,94.697017,2.350111,88.135204,0.719220,0.64900,0.63375,0.805,0.4750,0.1735,...,0.989959,0.985346,0.969900,0.959718,0.256181,0.281565,0.952306,0.921818,0.87830,0.85420
1,4.029266,92.250682,2.986979,75.154192,0.026955,0.07240,0.24520,0.425,0.4650,0.2795,...,0.978630,0.967782,0.983747,0.982618,0.310834,0.687882,0.793741,0.491959,0.88650,0.65725
2,4.194794,90.716169,2.284325,70.049311,0.014471,0.04975,0.17370,0.345,0.4625,0.2980,...,0.988265,0.983468,0.972389,0.969401,0.269742,0.359889,0.919112,0.835326,0.90400,0.81920
3,4.153689,90.213944,4.195769,65.280453,0.018105,0.04275,0.17640,0.365,0.4625,0.2725,...,0.980286,0.970285,0.977876,0.972328,0.296742,0.621278,0.795963,0.511964,0.87525,0.66700
4,2.905619,94.757097,2.507738,87.524032,0.732635,0.56800,0.62800,0.805,0.4750,0.2250,...,0.984807,0.977462,0.968727,0.957503,0.279793,0.480496,0.888070,0.734624,0.83325,0.72150


In [2]:
# test
design_df = pd.read_csv(f"{base}/snir/eval.csv")
X_test = design_df.reindex(columns=feature_cols)     # feature order
y_test = design_df["binder"]

print(f"test: {len(X_test)} rows, {X_test.shape[1]} features, classes={y_test.value_counts().to_dict()}")
X_test.head()

test: 35 rows, 49 features, classes={0: 32, 1: 3}


,af3_pae,af3_plddt,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq2,af3_lis,af3_iptm,af3_ptm,cf_af3_rmsd,...,boltz_template_ptm,boltz_template_iptm,boltz_template_plddt,boltz_template_iplddt,boltz_template_pde,boltz_template_ipde,boltz_template_per_chain_pair_iptm,boltz_template_ipsae,boltz_template_pdockq2,boltz_template_lis
0,2.593883,95.404666,1.993750,95.036725,0.788184,0.88085,0.75930,0.920,0.4800,0.1600,...,0.990514,0.986901,0.985055,0.985220,0.256930,0.278017,0.960036,0.937455,0.96685,0.87115
1,2.572969,95.599717,2.020938,94.909961,0.785245,0.89390,0.75510,0.915,0.4800,0.1665,...,0.989588,0.985429,0.987334,0.987299,0.258769,0.284754,0.952718,0.925141,0.96935,0.85870
2,3.486349,92.439163,5.111071,69.144182,0.088382,0.18020,0.34295,0.570,0.4700,0.1825,...,0.990548,0.986835,0.982311,0.979760,0.255680,0.283189,0.943557,0.906099,0.92390,0.82455
3,2.815847,95.282854,2.100000,94.730775,0.765295,0.88195,0.73785,0.910,0.4775,0.1800,...,0.991181,0.987563,0.979963,0.975488,0.256625,0.289870,0.943841,0.908519,0.92900,0.85605
4,2.790728,94.871984,2.275417,92.998114,0.769552,0.85065,0.74445,0.915,0.4775,0.1590,...,0.991275,0.987761,0.983894,0.983133,0.254981,0.275365,0.961354,0.939275,0.96220,0.87810


In [3]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, max_depth=5, class_weight="balanced", oob_score=True, random_state=42, n_jobs=-1) 
# min_samples_leaf: minimum number of samples required to be at a leaf node
# class_weight: The “balanced” mode uses the values of y to automatically adjust weights inversely proportional to class frequencies in the input data as n_samples / (n_classes * np.bincount(y))
# oob_score = use out-of-bag samples to estimate the generalization score.
# Popular integer random seeds are 0 and 42. 
# n_jobs = The number of jobs to run in parallel. -1 means using all processors.

In [4]:
# cross validation v1 (deafault "For int/None inputs, if the estimator is a classifier and y is either binary or multiclass, StratifiedKFold is used.")
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

cv_res = cross_validate(rf, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision"])
print(cv_res["test_roc_auc"].mean(), "±", cv_res["test_roc_auc"].std())
print(cv_res["test_average_precision"].mean(), "±", cv_res["test_average_precision"].std())

oof_probs = cross_val_predict(rf, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
print(confusion_matrix(y_train, (oof_probs >= 0.5).astype(int)))

0.8373148148148148 ± 0.03524799462137574
0.90753159648993 ± 0.02349141509017557
[[16 11]
 [ 9 35]]


In [5]:
# cross validation v2 --> grouped by first part of the sample name, maybe works better because of multiple mutants?
from sklearn.model_selection import StratifiedGroupKFold

groups = eval_df["sample"].str.split("_").str[0]       # <-- parent lineage, parent id?
n_splits = min(5, y_train.value_counts().min(), groups.nunique())

cv_group = StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)

cv_group_res = cross_validate(rf, X_train, y_train,  groups=groups, cv=cv_group, scoring=["roc_auc", "average_precision"])
print(cv_group_res["test_roc_auc"].mean(), "±", cv_group_res["test_roc_auc"].std())
print(cv_group_res["test_average_precision"].mean(), "±", cv_group_res["test_average_precision"].std())

oof_probs = cross_val_predict(rf, X_train, y_train, groups=groups, cv=cv_group, method="predict_proba")[:, 1]
print(confusion_matrix(y_train, (oof_probs >= 0.5).astype(int)))

0.8207142857142857 ± 0.06484491256694752
0.8842964623096201 ± 0.06839032557931286
[[13 14]
 [ 7 37]]


In [6]:
rf.fit(X_train, y_train)
print("OOB score:", rf.oob_score_)

OOB score: 0.676056338028169


In [7]:
design_df["p_binder"] = rf.predict_proba(X_test)[:, 1]   # probs
ranked = design_df.sort_values("p_binder", ascending=False)[["sample", "p_binder"]]   # sort descending rank
print(ranked.head(20))
# ranked.to_csv(f"{base}/design_rf_scores.csv", index=False)

                  sample  p_binder
1           5ig7_abc_cut  0.960649
17  5ig7_abc_cut_B_G114A  0.960649
0           5ifj_abc_cut  0.958097
3   5ifj_abc_cut_A_D111A  0.952728
15   5ig7_abc_cut_A_Y66A  0.951900
4   5ifj_abc_cut_A_K110A  0.949230
13  5ig7_abc_cut_A_K110A  0.948546
9   5ifj_abc_cut_B_L116A  0.943433
14   5ig7_abc_cut_A_W52A  0.942609
18  5ig7_abc_cut_B_L116A  0.940359
10   5ifj_abc_cut_B_V38A  0.940282
19   5ig7_abc_cut_B_V38A  0.928114
5    5ifj_abc_cut_A_S64A  0.926301
11  5ifj_abc_cut_B_W107A  0.912850
30   5ijk_acx_cut_B_W56A  0.904051
6    5ifj_abc_cut_A_W52A  0.901493
7    5ifj_abc_cut_A_Y66A  0.893205
26  5ijk_acx_cut_A_T113A  0.888618
34   5ijk_acx_cut_B_Y38A  0.857406
21   5ig7_abc_cut_B_Y37A  0.857247


In [8]:
# rf built in feature importance
rf_imp = (pd.DataFrame({"metric": X_train.columns,"importance": rf.feature_importances_}).sort_values("importance", ascending=False))

print(rf_imp)

                                metric  importance
13                           cf_iplddt    0.059715
34      boltz_free_per_chain_pair_iptm    0.042798
7                             af3_iptm    0.039879
37                      boltz_free_lis    0.039623
6                              af3_lis    0.034659
33                     boltz_free_ipde    0.032417
15                          cf_pdockq2    0.031888
16                              cf_lis    0.030581
17                             cf_iptm    0.029999
5                          af3_pdockq2    0.029332
4                            af3_ipsae    0.028280
40                 boltz_template_iptm    0.027686
20                           pred_lddt    0.027079
26                       esmfold2_iptm    0.025515
3                           af3_iplddt    0.025475
14                            cf_ipsae    0.024363
1                            af3_plddt    0.023597
47              boltz_template_pdockq2    0.022398
41                boltz_templat

In [9]:
# permutation feature importance (in progress)
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.inspection import permutation_importance

imps = []
for tr, te in cv.split(X_train, y_train):           # tr: train, te: hold out, for stratifiedgroupkfold: cv_group, add: groups=groups
    m = clone(rf).fit(X_train.iloc[tr], y_train.iloc[tr])          # fit on the training folds
    r = permutation_importance(m, X_train.iloc[te], y_train.iloc[te], scoring="average_precision", n_repeats=10, random_state=42, n_jobs=1)  # compute on hold out
    imps.append(r.importances_mean)

imp = (pd.DataFrame(imps, columns=X_train.columns).mean().sort_values(ascending=False).rename_axis("metric").rename("importance").reset_index())
print(imp)

                                metric  importance
0                   boltz_template_ptm    0.010365
1               boltz_template_pdockq2    0.010263
2                       boltz_free_pde    0.008236
3                              af3_lis    0.008097
4                           cf_pdockq2    0.007069
5   boltz_template_per_chain_pair_iptm    0.006612
6                            cf_iplddt    0.006285
7                      boltz_free_ipde    0.005996
8                            af3_ipsae    0.004765
9                          af3_pdockq2    0.004600
10                  boltz_template_lis    0.004206
11                boltz_template_plddt    0.004027
12         boltz_free_confidence_score    0.003907
13                            cf_ipsae    0.003326
14                 boltz_template_ipde    0.003229
15               boltz_template_iplddt    0.003145
16                             cf_ipae    0.003094
17      boltz_free_per_chain_pair_iptm    0.002787
18                           pr